# YOLO26 跨 GPU Runtime Prediction：實驗 Notebook 導覽

本目錄將既有 Part 1–8 實驗整理成可閱讀、可檢查與可選擇重跑的 Notebook。原始腳本、資料與既有輸出均保留在原位置。

## 使用原則

- Notebook 預設只讀取既有結果，不重跑昂貴分析。
- 每份 Notebook 的 `RUN_EXPENSIVE = False` 是安全開關；只有確認環境、輸入與輸出位置後才改成 `True`。
- 既有報告會以快照嵌入 Notebook，因此未執行 cell 也能閱讀主要結果。
- Part 1–6 多半是 oracle／marker-aligned benchmark；Part 7–8 才逐步接近 marker-free deployment。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 研究演進

| Part | 核心問題 | 部署定位 |
|---:|---|---|
| 1 | source runtime 是否能預測 target runtime | feasibility baseline |
| 2 | workload metadata 是否改善 scaling | idealized、需要 metadata |
| 3 | GPU behavior 是否足以預測 | marker-aligned oracle |
| 4 | trace／HPCToolkit proxy 是否有增益 | marker-aligned ablation |
| 5 | 模型封裝、不確定性與 slice | oracle model engineering |
| 6 | 短 prefix 是否足夠 | short-horizon oracle |
| 7 | 無 marker 找週期 | deployment-safe detector |
| 8 | 無 marker 預測 target runtime | 目前主要部署估計 |

In [ ]:
core_inputs = [
    "analysis_outputs/collection_inventory.csv",
    "analysis_outputs/ground_truth.csv",
    "hardware/hardware_descriptors.csv",
    "configs/experiments.csv",
]
display(inspect_inputs(core_inputs))
display(load_csv("analysis_outputs/ground_truth.csv").head())
display(load_csv("hardware/hardware_descriptors.csv"))

## 建議閱讀順序

依序閱讀 Part 1–8，最後開啟 99_final_results_summary.ipynb 比較研究演進。若只關心目前部署結果，可直接閱讀 Part 7、Part 8 與總結。